# DegreeDetailExtract — v3 Anti-Overfitting Training Notebook

**What's new in v3** (vs v2):
- 36 total certificate layout templates (22 original + 6 v2 + **8 new v3**)
- Rich ordinal/textual date formats: *"15th day of June, 2002"*, *"March 11th, 2008"*, etc.
- Optional pass_class — ~30% of generated certs have no class (like real ones)
- 16 prose styles including fully-cursive body, name-at-top, label-value grid, gothic
- **12 training epochs** (was 4) with cosine LR schedule — model actually converges
- Per-epoch checkpoints → resume cell if Colab disconnects
- Per-field accuracy logged every epoch
- 5,500 generated certificates (was 3,000)


## Section 0 — Environment Setup


In [ ]:
# ── Step 1: system fonts ────────────────────────────────────────────────
!apt-get install -y fonts-liberation fonts-dejavu-core -qq

# ── Step 2: tokenizers (pre-built binary wheel — avoids Rust build) ────
# Must be installed BEFORE transformers to prevent source compilation.
!pip install -q --prefer-binary tokenizers

# ── Step 3: transformers + training stack ───────────────────────────────
# No strict version pin on transformers — Colab ships a compatible version.
!pip install -q transformers datasets sentencepiece

# ── Step 4: certificate generation stack ────────────────────────────────
!pip install -q Pillow tqdm faker albumentations opencv-python-headless

# ── Step 5: evaluation tools ────────────────────────────────────────────
!pip install -q editdistance scikit-learn

print('\n✅ All packages installed.')

# Sanity-check critical imports
import transformers, tokenizers, faker, albumentations
print(f'   transformers: {transformers.__version__}')
print(f'   tokenizers  : {tokenizers.__version__}')


In [ ]:
import torch
cuda_ok = torch.cuda.is_available()
print('CUDA:', cuda_ok)
if cuda_ok:
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('WARNING: No GPU — training will be extremely slow!')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/DegreeDetailExtract', exist_ok=True)
print('Google Drive mounted.')


## Section 1 — Clone Repo & Download Fonts


In [ ]:
import os, sys

REPO = '/content/DegreeDetailExtract'
if not os.path.exists(REPO):
    !git clone https://github.com/Vrushti33/DegreeDetailExtract.git {REPO}
else:
    !git -C {REPO} pull

sys.path.insert(0, REPO)

real_certs = os.listdir(f'{REPO}/real_certs')
print(f'Real certs in repo: {len(real_certs)}')
for f in real_certs:
    print(' ', f)


In [ ]:
# Download calligraphy / formal certificate fonts from Google Fonts (OFL)
#
#  Great Vibes        — student name in flowing script
#  Cinzel [wght]      — university name in engraved Roman capitals
#  IM Fell English    — body prose in antique book type
#  Playfair Display   — section headings (high-contrast serif)
#  UnifrakturMaguntia — optional Old-English blackletter title

from generator.fonts import download_certificate_fonts
download_certificate_fonts('/content/cert_fonts')
print('\n✅ Calligraphy fonts ready.')


## Section 2 — Generate 5,500 Realistic v3 Certificates

Expected time: **18–28 min** on a T4 GPU Colab session.
Generates 36-template certificates with real paper textures, ordinal dates, and prose.


In [ ]:
# ── Generate 5,500 realistic v3 certificates ─────────────────────────────
# Running directly with ! allows tqdm progress bar to render LIVE in Colab.
import os, sys

REPO       = '/content/DegreeDetailExtract'
DATASET    = '/content/dataset_v3'
REAL_CERTS = f'{REPO}/real_certs'

print('Starting certificate generation (5,500 images)...')
print('Progress bar updates live below (~20-25 min on T4 GPU):\n')

# Run generation command (live progress display)
!cd {REPO} && python generate_certificates_v3.py \
    --count 5500 \
    --output_dir {DATASET} \
    --real_certs_dir {REAL_CERTS} \
    --seed 42

# Validate output
print('\nDataset stats:')
ok = True
for split in ('train', 'val', 'test'):
    meta_path = f'{DATASET}/metadata_{split}.jsonl'
    if os.path.exists(meta_path):
        n = sum(1 for _ in open(meta_path))
        print(f'  {split:5s}: {n:,} records')
    else:
        print(f'  {split:5s}: MISSING')
        ok = False

if not ok:
    raise RuntimeError('One or more metadata files are missing. '
                       'Generation likely failed.')

print('\n✅ Generation complete!')


In [ ]:
# Preview 8 random v3 certificates to visually verify realism
import random, json
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 80

DATASET = '/content/dataset_v3'
with open(f'{DATASET}/metadata_train.jsonl') as f:
    rows = [json.loads(l) for l in f]

samples = random.sample(rows, min(8, len(rows)))
fig, axes = plt.subplots(2, 4, figsize=(20, 14))
for ax, row in zip(axes.flat, samples):
    img = Image.open(f"{DATASET}/{row['file_name']}")
    ax.imshow(img)
    pc = row.get('pass_class', '')
    title = f"{row['student_name'][:18]}\n{row['course_name'][:20]}\n{row['issue_date'][:16]}"
    if pc:
        title += f'\n{pc}'
    ax.set_title(title, fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('\nVerify above: real textures, calligraphy fonts, ceremonial prose, ordinal dates, varied spacing.')


In [ ]:
# Zip and save dataset to Google Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/DegreeDetailExtract'
DATASET = '/content/dataset_v3'
zip_path = f'{DRIVE}/dataset_v3_archive'
print('Zipping dataset...')
shutil.make_archive(zip_path, 'zip', DATASET)
size_gb = os.path.getsize(zip_path + '.zip') / 1e9
print(f'Saved: {zip_path}.zip  ({size_gb:.2f} GB)')


## Section 3 — Fine-tune Donut (12 Epochs)

> **Expected time**: ~75–100 min on T4 GPU for 12 epochs over 5,500 images.
>
> Checkpoints are saved to Google Drive **after every epoch**.
> If Colab disconnects, skip to **Section 4 (Resume)** to continue.


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

from transformers import DonutProcessor, VisionEncoderDecoderModel
import os

DRIVE_CKPTS = '/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v3'
os.makedirs(DRIVE_CKPTS, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'naver-clova-ix/donut-base'

processor = DonutProcessor.from_pretrained(MODEL_ID)
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)

# Special tokens for all 7 fields + s/e markers
NEW_TOKENS = [
    '<s_cert>', '</s_cert>',
    '<s_student_name>',    '</s_student_name>',
    '<s_university_name>', '</s_university_name>',
    '<s_course_name>',     '</s_course_name>',
    '<s_specialization>',  '</s_specialization>',
    '<s_pass_class>',      '</s_pass_class>',
    '<s_authority_name>',  '</s_authority_name>',
    '<s_issue_date>',      '</s_issue_date>',
]
processor.tokenizer.add_special_tokens({'additional_special_tokens': NEW_TOKENS})
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s_cert>'])[0]
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.convert_tokens_to_ids(['</s_cert>'])[0]

model = model.to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model loaded ({n_params:.0f}M params) on {device}')


In [ ]:
import json, os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch

DATASET = '/content/dataset_v3'

FIELDS = ['student_name', 'university_name', 'course_name',
          'specialization', 'pass_class', 'authority_name', 'issue_date']

def build_target(row):
    """Build Donut target string. Omit pass_class tag if empty."""
    parts = ['<s_cert>']
    for f in FIELDS:
        v = row.get(f, '')
        if f == 'pass_class' and not v:
            # Explicitly encode as empty so model learns to predict empty
            parts.append(f'<s_pass_class></s_pass_class>')
        else:
            parts.append(f'<s_{f}>{v}</{f[0]+f[1:]}>'  # e.g. <s_student_name>...
                         .replace(f'</{f[0]+f[1:]}>', f'</{f}>'))
    parts.append('</s_cert>')
    return ''.join(parts)


class CertDataset(Dataset):
    def __init__(self, split):
        with open(f'{DATASET}/metadata_{split}.jsonl') as fp:
            self.rows = [json.loads(l) for l in fp]
        self.split = split

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img  = Image.open(f"{DATASET}/{row['file_name']}").convert('RGB')
        pixel_values = processor(
            images=img, return_tensors='pt'
        ).pixel_values.squeeze(0)
        target = build_target(row)
        labels = processor.tokenizer(
            target, add_special_tokens=False,
            max_length=512, truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze(0)
        return pixel_values, labels


def collate_fn(batch):
    pixels, labels_list = zip(*batch)
    pixels = torch.stack(pixels)
    max_len = max(l.size(0) for l in labels_list)
    pad = processor.tokenizer.pad_token_id
    padded = torch.full((len(labels_list), max_len), pad, dtype=torch.long)
    for i, l in enumerate(labels_list):
        padded[i, :l.size(0)] = l
    padded[padded == pad] = -100  # ignore padding in loss
    return pixels, padded


train_ds = CertDataset('train')
val_ds   = CertDataset('val')
test_ds  = CertDataset('test')

BATCH = 1          # Batch size 1 + Grad Accum 8 fits comfortably in T4 VRAM
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      num_workers=2, collate_fn=collate_fn, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                      num_workers=2, collate_fn=collate_fn, pin_memory=True)

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  Training — 12 epochs, FP16 + Gradient Checkpointing (T4 Safe)
# ─────────────────────────────────────────────────────────────────────
import os, math, json, sys, gc, torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import editdistance

# Release any lingering references from previous failed cell runs
sys.last_traceback = None
sys.last_value = None
gc.collect()
torch.cuda.empty_cache()

# Enable Gradient Checkpointing on Donut — slashes activation VRAM by >60%
model.gradient_checkpointing_enable()

NUM_EPOCHS       = 12
LR               = 3e-5
WARMUP_STEPS     = 200
GRAD_ACCUM       = 8          # effective batch = 1 * 8 = 8
MAX_GRAD_NORM    = 1.0
LOG_EVERY        = 50
EVAL_EVERY_EPOCH = True
DRIVE_CKPTS      = '/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v3'

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_dl) * NUM_EPOCHS // GRAD_ACCUM
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')  # Modern PyTorch AMP

# Warmup wrapper
step_global = 0
def get_lr():
    if step_global < WARMUP_STEPS:
        return LR * step_global / max(1, WARMUP_STEPS)
    return scheduler.get_last_lr()[0]


def evaluate_fields(model, dl, max_batches=30):
    """Compute per-field exact-match accuracy on val set."""
    model.eval()
    field_correct = {f: 0 for f in FIELDS}
    field_total   = {f: 0 for f in FIELDS}

    with torch.no_grad():
        for b_idx, (pixels, _) in enumerate(dl):
            if b_idx >= max_batches:
                break
            pixels = pixels.to(device)
            task_prompt = processor.tokenizer.decode(
                [model.config.decoder_start_token_id], skip_special_tokens=False
            )
            decoder_ids = processor.tokenizer(
                task_prompt, add_special_tokens=False, return_tensors='pt'
            ).input_ids.to(device)
            decoder_ids = decoder_ids.repeat(pixels.size(0), 1)

            with torch.amp.autocast('cuda'):
                outputs = model.generate(
                    pixel_values=pixels,
                    decoder_input_ids=decoder_ids,
                    max_new_tokens=512,
                    early_stopping=True,
                )
            decoded = processor.tokenizer.batch_decode(outputs, skip_special_tokens=False)

            for pred_str in decoded:
                for f in FIELDS:
                    import re
                    m = re.search(fr'<s_{f}>(.*?)</{f}>', pred_str, re.DOTALL)
                    field_total[f] += 1
                    if m:
                        field_correct[f] += 1

    model.train()
    return {f: field_correct[f] / max(1, field_total[f]) for f in FIELDS}


def save_checkpoint(epoch, val_loss):
    ckpt_dir = f'{DRIVE_CKPTS}/epoch_{epoch:02d}'
    os.makedirs(ckpt_dir, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    meta = {'epoch': epoch, 'val_loss': val_loss,
            'step': step_global, 'lr': get_lr()}
    with open(f'{ckpt_dir}/train_meta.json', 'w') as fp:
        json.dump(meta, fp, indent=2)
    print(f'  [ckpt] Saved epoch {epoch} -> {ckpt_dir}  (val_loss={val_loss:.4f})')


print(f'Starting v3 training: {NUM_EPOCHS} epochs | FP16 + Gradient Checkpointing | LR {LR} -> 1e-6 cosine')
print(f'  Batch size: 1 | Grad accum: {GRAD_ACCUM} (Effective batch: {1 * GRAD_ACCUM})')
print(f'  Total steps: ~{total_steps:,}')
print(f'  Checkpoints: {DRIVE_CKPTS}\n')

history = []
best_val_loss = float('inf')
patience_count = 0
EARLY_STOP_PATIENCE = 4

model.train()
for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\n=== Epoch {epoch}/{NUM_EPOCHS} ===')
    train_loss = 0.0
    optimizer.zero_grad()

    for batch_idx, (pixels, labels) in enumerate(train_dl):
        pixels = pixels.to(device)
        labels = labels.to(device)

        if step_global < WARMUP_STEPS:
            for pg in optimizer.param_groups:
                pg['lr'] = LR * step_global / max(1, WARMUP_STEPS)

        # FP16 Forward pass with autocast
        with torch.amp.autocast('cuda'):
            out  = model(pixel_values=pixels, labels=labels)
            loss = out.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        train_loss += loss.item() * GRAD_ACCUM

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            if step_global >= WARMUP_STEPS:
                scheduler.step()
            optimizer.zero_grad()
            step_global += 1

        if (batch_idx + 1) % LOG_EVERY == 0:
            avg = train_loss / (batch_idx + 1)
            print(f'  step {step_global:5d}  batch {batch_idx+1}/{len(train_dl)}'
                  f'  loss={avg:.4f}  lr={get_lr():.2e}')

    avg_train_loss = train_loss / len(train_dl)

    # Validation loss
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for pixels, labels in val_dl:
            pixels, labels = pixels.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                val_loss += model(pixel_values=pixels, labels=labels).loss.item()
    val_loss /= len(val_dl)
    model.train()

    # Per-field accuracy
    field_acc = evaluate_fields(model, val_dl, max_batches=30)

    print(f'  Epoch {epoch:2d}: train_loss={avg_train_loss:.4f}  val_loss={val_loss:.4f}')
    print('  Field accuracy:')
    for f, acc in field_acc.items():
        bar = '|' * int(acc * 20)
        print(f'    {f:22s}: {acc*100:5.1f}%  {bar}')

    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'val_loss': val_loss})
    save_checkpoint(epoch, val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        best_dir = f'{DRIVE_CKPTS}/best'
        import shutil
        if os.path.exists(best_dir):
            shutil.rmtree(best_dir)
        shutil.copytree(f'{DRIVE_CKPTS}/epoch_{epoch:02d}', best_dir)
        print(f'  [best] New best val_loss={best_val_loss:.4f} saved.')
    else:
        patience_count += 1
        if patience_count >= EARLY_STOP_PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs).')
            break

print('\nTraining complete.')


## Section 4 — Resume Training (if Colab Disconnected)

Run this cell to resume from the **latest checkpoint** saved to Drive.
Skip this section entirely if training completed normally in Section 3.


In [ ]:
# Detect latest epoch checkpoint in Drive and resume training
import os, json, glob, shutil, torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import DonutProcessor, VisionEncoderDecoderModel

DRIVE_CKPTS = '/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v3'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Find latest epoch checkpoint
epoch_dirs = sorted(glob.glob(f'{DRIVE_CKPTS}/epoch_*'))
if not epoch_dirs:
    raise RuntimeError('No checkpoints found. Run Section 3 first.')

latest_dir = epoch_dirs[-1]
with open(f'{latest_dir}/train_meta.json') as fp:
    meta = json.load(fp)

start_epoch = meta['epoch'] + 1
step_global  = meta['step']

print(f'Resuming from: {latest_dir}')
print(f'  Completed epochs: {meta["epoch"]}  |  Resume from epoch: {start_epoch}')
print(f'  Global step so far: {step_global}')

# Reload model
import sys
sys.path.insert(0, '/content/DegreeDetailExtract')
processor = DonutProcessor.from_pretrained(latest_dir)
model     = VisionEncoderDecoderModel.from_pretrained(latest_dir).to(device)

# --- rebuild dataset (re-run Section 3 dataset cell if needed) ---
# (assumes train_dl / val_dl already exist from the current session)

NUM_EPOCHS   = 12
LR           = 3e-5
WARMUP_STEPS = 200
GRAD_ACCUM   = 4          # effective batch = 2 * 4 = 8
MAX_GRAD_NORM= 1.0
LOG_EVERY    = 50
EARLY_STOP_PATIENCE = 4
DRIVE_CKPTS  = '/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v3'

total_steps   = len(train_dl) * NUM_EPOCHS // GRAD_ACCUM
optimizer     = AdamW(model.parameters(), lr=meta['lr'], weight_decay=0.01)
scheduler     = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)
scaler        = torch.cuda.amp.GradScaler()

best_val_loss = meta['val_loss']
patience_count = 0

print(f'\nResumed. Training epochs {start_epoch}–{NUM_EPOCHS}...')
model.train()

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print(f'\n=== Epoch {epoch}/{NUM_EPOCHS} (resumed) ===')
    train_loss = 0.0
    optimizer.zero_grad()

    for batch_idx, (pixels, labels) in enumerate(train_dl):
        pixels, labels = pixels.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            out  = model(pixel_values=pixels, labels=labels)
            loss = out.loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        train_loss += loss.item() * GRAD_ACCUM

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step_global += 1

        if (batch_idx + 1) % LOG_EVERY == 0:
            print(f'  step {step_global}  loss={train_loss/(batch_idx+1):.4f}')

    avg_train_loss = train_loss / len(train_dl)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for pixels, labels in val_dl:
            pixels, labels = pixels.to(device), labels.to(device)
            val_loss += model(pixel_values=pixels, labels=labels).loss.item()
    val_loss /= len(val_dl)
    model.train()

    print(f'  Epoch {epoch}: train={avg_train_loss:.4f}  val={val_loss:.4f}')

    # Save checkpoint
    ckpt_dir = f'{DRIVE_CKPTS}/epoch_{epoch:02d}'
    os.makedirs(ckpt_dir, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    with open(f'{ckpt_dir}/train_meta.json', 'w') as fp:
        json.dump({'epoch': epoch, 'val_loss': val_loss, 'step': step_global, 'lr': LR}, fp)
    print(f'  [ckpt] epoch {epoch} saved.')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        best_dir = f'{DRIVE_CKPTS}/best'
        if os.path.exists(best_dir): shutil.rmtree(best_dir)
        shutil.copytree(ckpt_dir, best_dir)
        print(f'  [best] saved.')
    else:
        patience_count += 1
        if patience_count >= EARLY_STOP_PATIENCE:
            print('Early stopping.')
            break

print('Resume complete.')


## Section 5 — Evaluation


In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch

DRIVE_CKPTS = '/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v3'
BEST = f'{DRIVE_CKPTS}/best'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = DonutProcessor.from_pretrained(BEST)
model     = VisionEncoderDecoderModel.from_pretrained(BEST).to(device)
model.eval()
print('Best model loaded from:', BEST)


In [ ]:
import json, re, editdistance
from PIL import Image
import torch

DATASET = '/content/dataset_v3'
FIELDS  = ['student_name', 'university_name', 'course_name',
           'specialization', 'pass_class', 'authority_name', 'issue_date']

with open(f'{DATASET}/metadata_test.jsonl') as fp:
    test_rows = [json.loads(l) for l in fp]

MAX_EVAL = min(200, len(test_rows))
field_exact  = {f: 0 for f in FIELDS}
field_cer    = {f: [] for f in FIELDS}  # character error rate
field_total  = {f: 0 for f in FIELDS}

task_prompt = processor.tokenizer.decode(
    [model.config.decoder_start_token_id], skip_special_tokens=False)

for row in test_rows[:MAX_EVAL]:
    img = Image.open(f"{DATASET}/{row['file_name']}").convert('RGB')
    pixels = processor(images=img, return_tensors='pt').pixel_values.to(device)
    dec_ids = processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors='pt'
    ).input_ids.to(device)
    with torch.no_grad():
        out = model.generate(pixel_values=pixels, decoder_input_ids=dec_ids,
                             max_new_tokens=512, early_stopping=True)
    pred_str = processor.tokenizer.decode(out[0], skip_special_tokens=False)

    for f in FIELDS:
        gt  = str(row.get(f, ''))
        m   = re.search(fr'<s_{f}>(.*?)</{f}>', pred_str, re.DOTALL)
        pred = m.group(1).strip() if m else ''
        field_total[f] += 1
        if pred.lower() == gt.lower():
            field_exact[f] += 1
        cer = editdistance.eval(pred, gt) / max(len(gt), 1)
        field_cer[f].append(cer)

print(f'Evaluation on {MAX_EVAL} test certificates\n')
print(f'{"Field":<25} {"Exact %":>8} {"Avg CER":>8}')
print('-' * 45)
for f in FIELDS:
    exact_pct = field_exact[f] / max(1, field_total[f]) * 100
    avg_cer   = sum(field_cer[f]) / max(1, len(field_cer[f])) * 100
    bar = '|' * int(exact_pct / 5)
    print(f'{f:<25} {exact_pct:7.1f}%  {avg_cer:7.1f}%   {bar}')


In [ ]:
# Show 3 test examples: predicted fields vs ground truth
import random, re, json
from PIL import Image
import matplotlib.pyplot as plt

DATASET = '/content/dataset_v3'
FIELDS  = ['student_name', 'university_name', 'course_name',
           'specialization', 'pass_class', 'authority_name', 'issue_date']

with open(f'{DATASET}/metadata_test.jsonl') as fp:
    test_rows = [json.loads(l) for l in fp]

samples = random.sample(test_rows, 3)
fig, axes = plt.subplots(1, 3, figsize=(18, 9))

task_prompt = processor.tokenizer.decode(
    [model.config.decoder_start_token_id], skip_special_tokens=False)

for ax, row in zip(axes, samples):
    img = Image.open(f"{DATASET}/{row['file_name']}").convert('RGB')
    pixels = processor(images=img, return_tensors='pt').pixel_values.to(device)
    dec_ids = processor.tokenizer(task_prompt, add_special_tokens=False,
                                   return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        out = model.generate(pixel_values=pixels, decoder_input_ids=dec_ids,
                             max_new_tokens=512, early_stopping=True)
    pred_str = processor.tokenizer.decode(out[0], skip_special_tokens=False)

    annotation = ''
    for f in FIELDS:
        gt = str(row.get(f, ''))
        m  = re.search(fr'<s_{f}>(.*?)</{f}>', pred_str, re.DOTALL)
        pred = m.group(1).strip() if m else 'MISSING'
        icon = 'OK' if pred.lower() == gt.lower() else 'X'
        annotation += f'[{icon}] {f[:14]}: {pred[:22]}\n'

    ax.imshow(img)
    ax.set_title(annotation, fontsize=6.5, family='monospace', loc='left')
    ax.axis('off')

plt.tight_layout()
plt.show()


## Section 6 — Inference on a Real Certificate


In [ ]:
from google.colab import files
uploaded = files.upload()
real_cert_path = list(uploaded.keys())[0]
print('Uploaded:', real_cert_path)


In [ ]:
import re, json
from PIL import Image
import torch

FIELDS = ['student_name', 'university_name', 'course_name',
          'specialization', 'pass_class', 'authority_name', 'issue_date']

img = Image.open(real_cert_path).convert('RGB')
pixels = processor(images=img, return_tensors='pt').pixel_values.to(device)

task_prompt = processor.tokenizer.decode(
    [model.config.decoder_start_token_id], skip_special_tokens=False)
dec_ids = processor.tokenizer(task_prompt, add_special_tokens=False,
                               return_tensors='pt').input_ids.to(device)

with torch.no_grad():
    out = model.generate(pixel_values=pixels, decoder_input_ids=dec_ids,
                         max_new_tokens=512, early_stopping=True)

pred_str = processor.tokenizer.decode(out[0], skip_special_tokens=False)

result = {}
for f in FIELDS:
    m = re.search(fr'<s_{f}>(.*?)</{f}>', pred_str, re.DOTALL)
    result[f] = m.group(1).strip() if m else ''

print(json.dumps(result, indent=2, ensure_ascii=False))

missing = [f for f in FIELDS if f not in result or not result[f]]
if not missing:
    print('\nAll 7 fields extracted.')
else:
    print('\nMissing fields:', missing)
